# Conexiones e importaciones


In [1]:
import os #trabajar con el sistema operativo y variables de entorno
from dotenv import load_dotenv #cargar variable de entorno
import pandas as pd ## para convertir los datos en formato tabla
import mysql.connector ## para conectar Paython con MySQL
import numpy as np ## para convertir formato nan de Paython a none de MySQL
from mysql.connector import Error ## captura errores de MySQL

load_dotenv()

MYSQL_HOST = os.getenv("MYSQL_HOST")
MYSQL_USER = os.getenv("MYSQL_USER")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD")
DB_NAME = "ABC_Corporation_db"


# Funciones para la conexión con Mysql, crear base de datos e insertar datos

In [2]:
# FUNCIÓN PARA CONECTAR CON MYSQL

def connect_mysql():
    try:
        cnx = mysql.connector.connect(
            host=MYSQL_HOST,
            user=MYSQL_USER,
            password=MYSQL_PASSWORD
        )
        print("Conexión a MySQL exitosa")
        return cnx
    except Error as e:
        print("Error al conectar a MySQL:", e)
        return None


In [3]:
# FUNCIÓN PARA CREAR BASE DATOS

def create_database(cnx):
    try:
        mycursor = cnx.cursor()
        mycursor.execute(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")
        print("Base de datos creada")
    except Error as e:
        print("Error al crear la base de datos:", e)


In [4]:
# FUNCIÓN PARA CREAR TABLAS EN LA BASE DE DATOS

def create_tables(cnx):
    # crea cursor para ejecutar comandos SQL
    mycursor = cnx.cursor() 
    # indica la base de datos donde vamos a crear las tablas
    mycursor.execute("USE ABC_Corporation_db") 

    # Creación de una lista donde cada elemento es una query para crear cada tabla
    tables = [
        
        # Tabla Deparments
        """CREATE TABLE IF NOT EXISTS Departments (
            ID_Department INT AUTO_INCREMENT PRIMARY KEY,
            Department VARCHAR(50)
        );""",
        # Tabla JobRoles
        """CREATE TABLE IF NOT EXISTS JobRoles (
            ID_JobRole INT AUTO_INCREMENT PRIMARY KEY,
            JobRole VARCHAR(50)
        );""",
        # Tabla Employees
        """CREATE TABLE IF NOT EXISTS Employees (
            EmployeeNumber INT PRIMARY KEY,
            Age FLOAT,
            AgeGroup VARCHAR(20),
            Gender VARCHAR(20),
            MaritalStatus VARCHAR(20),
            DistanceFromHome INT,
            Education INT,
            EducationField VARCHAR(50),
            JobLevel INT,
            Attrition VARCHAR(10),
            Attrition_flag INT,
            ID_Department INT,
            JobRole VARCHAR(50),
            FOREIGN KEY (ID_Department) REFERENCES Departments(ID_Department)
        );""",
        # Tabla Salaries
        """CREATE TABLE IF NOT EXISTS Salaries (
            EmployeeNumber INT PRIMARY KEY,
            HourlyRate INT,
            DailyRate INT,
            MonthlyRate INT,
            MonthlyIncome FLOAT,
            PercentSalaryHike INT,
            StockOptionLevel INT,
            FOREIGN KEY (EmployeeNumber) REFERENCES Employees(EmployeeNumber)
        );""",
        # Tabla Satisfaction
        """CREATE TABLE IF NOT EXISTS Satisfaction (
            EmployeeNumber INT PRIMARY KEY,
            EnvironmentSatisfaction INT,
            JobSatisfaction INT,
            RelationshipSatisfaction INT,
            WorkLifeBalance INT,
            FOREIGN KEY (EmployeeNumber) REFERENCES Employees(EmployeeNumber)
        );""",
        # Tabla WorkConditions
        """CREATE TABLE IF NOT EXISTS WorkConditions (
            EmployeeNumber INT PRIMARY KEY,
            BusinessTravel VARCHAR(30),
            OverTime VARCHAR(10),
            JobInvolvement INT,
            PerformanceRating INT,
            FOREIGN KEY (EmployeeNumber) REFERENCES Employees(EmployeeNumber)
        );""",
        # Tabla Experience
        """CREATE TABLE IF NOT EXISTS Experience (
            EmployeeNumber INT PRIMARY KEY,
            TotalWorkingYears INT,
            NumCompaniesWorked INT,
            YearsAtCompany INT,
            YearsInCurrentRole INT,
            YearsSinceLastPromotion INT,
            YearsWithCurrManager FLOAT,
            TrainingTimesLastYear FLOAT,
            FOREIGN KEY (EmployeeNumber) REFERENCES Employees(EmployeeNumber)
        );"""
    ]

    try:
        # Ejecuta cada query para crear tabla
        for table in tables:
            mycursor.execute(table)
        cnx.commit()
        print("Todas las tablas creadas correctamente")
    except Exception as e:
        cnx.rollback()
        print("Error al crear las tablas:", e)


In [5]:
# FUNCIÓN PARA LEER ARCHIVO

def read_dataset(ruta):
    
    try:
        if ruta.endswith('.csv'): # comprueba si esl fichero termina en csv
            df = pd.read_csv(ruta)
        elif ruta.endswith(('.xls', '.xlsx')):
            df = pd.read_excel(ruta)
        else:
            print(" Tipo de fichero no soportado. Solo CSV o Excel.")
            return None

        return df

    except Exception as e:
        print(" Error al leer el fichero:", e)
        return None


In [6]:
# FUNCIÓN PARA INSERTAR DATOS 

def insert_data(cnx, df):
    mycursor = cnx.cursor()
    mycursor.execute("USE ABC_Corporation_db")

    try:
        # 1. Insertar Departments
        departments = df[['Department']].drop_duplicates().values.tolist()
        mycursor.executemany(
            "INSERT INTO Departments (Department) VALUES (%s)",
            departments
        )

        # 2. Insertar JobRoles
        jobroles = df[['JobRole']].drop_duplicates().values.tolist()
        mycursor.executemany(
            "INSERT INTO JobRoles (JobRole) VALUES (%s)",
            jobroles
        )

        # 3. Mapear ID_Department (Foreign Key)
        mycursor.execute("SELECT ID_Department, Department FROM Departments")
        dept_map = {name: id_ for id_, name in mycursor.fetchall()}
        df['ID_Department'] = df['Department'].map(dept_map)

        # 4. Insertar Employees
        employees_cols = [
            'EmployeeNumber','Age','AgeGroup','Gender','MaritalStatus',
            'DistanceFromHome','Education','EducationField','JobLevel',
            'Attrition','Attrition_flag','ID_Department','JobRole'
        ]
        mycursor.executemany(
            """INSERT INTO Employees VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)""",
            df[employees_cols].values.tolist()
        )

        # 5. Insertar Salaries
        salaries_cols = [
            'EmployeeNumber','HourlyRate','DailyRate','MonthlyRate',
            'MonthlyIncome','PercentSalaryHike','StockOptionLevel'
        ]
        mycursor.executemany(
            """INSERT INTO Salaries VALUES (%s,%s,%s,%s,%s,%s,%s)""",
            df[salaries_cols].values.tolist()
        )

        # 6. Insertar Satisfaction
        satisfaction_cols = [
            'EmployeeNumber','EnvironmentSatisfaction','JobSatisfaction',
            'RelationshipSatisfaction','WorkLifeBalance'
        ]
        mycursor.executemany(
            """INSERT INTO Satisfaction VALUES (%s,%s,%s,%s,%s)""",
            df[satisfaction_cols].values.tolist()
        )

        # 7. Insertar WorkConditions
        work_cols = [
            'EmployeeNumber','BusinessTravel','OverTime',
            'JobInvolvement','PerformanceRating'
        ]
        mycursor.executemany(
            """INSERT INTO WorkConditions VALUES (%s,%s,%s,%s,%s)""",
            df[work_cols].values.tolist()
        )

        # 8. Insertar Experience
        experience_cols = [
            'EmployeeNumber','TotalWorkingYears','NumCompaniesWorked',
            'YearsAtCompany','YearsInCurrentRole',
            'YearsSinceLastPromotion','YearsWithCurrManager',
            'TrainingTimesLastYear'
        ]
        mycursor.executemany(
            """INSERT INTO Experience VALUES (%s,%s,%s,%s,%s,%s,%s,%s)""",
            df[experience_cols].values.tolist()
        )

        cnx.commit()
        print("Todos los datos insertados correctamente")

    except Exception as e:
        cnx.rollback()
        print("Error al insertar datos:", e)


In [7]:
# FUNCIÓN PARA CERRAR CONEXION

def close_connection(cnx):
    try:
        if cnx is not None and cnx.is_connected():
            cnx.close()
            print("Conexión a MySQL cerrada correctamente")
    except Exception as e:
        print("Error al cerrar la conexión:", e)


# Ejecutar funciones

In [8]:
cnx = connect_mysql()


Conexión a MySQL exitosa


In [9]:
create_database(cnx)


Base de datos creada


In [10]:
create_tables(cnx)


Todas las tablas creadas correctamente


In [11]:
df_final=read_dataset("df_final.csv")

In [12]:
# Leer fichero
df_final = pd.read_csv("df_final.csv")


In [13]:
insert_data(cnx, df_final)


Todos los datos insertados correctamente


In [14]:
close_connection(cnx)

Conexión a MySQL cerrada correctamente
